# Saunders b10 — SpatialProp vs CONCERT (PCC-delta + composition Overlap)

End-to-end spatial perturbation benchmark on **Saunders 2025 MERFISH liver, slice `Batch_10_Slice_0`** (67512 cells × 209 genes, 9 cell types), 11 in-panel guides.

Two **complementary** scoring currencies on the identical predicted niches:
- **PCC-delta** — does the prediction move the right *genes* in the right direction (expression shift)? Scored in **seed** (perturbed centre cell) and **niche** (bystander neighbours) dimensions.
- **composition Overlap** *(new)* — does the prediction reproduce the right *cell-type composition* of the niche? `Overlap = 1 - TV ∈ [0,1]`, read as 'what fraction of the niche cell-type makeup is predicted right'. This is the spatially-native readout PCC-delta is blind to (PCC-delta is permutation-invariant over cells — it ignores who sits next to whom).

All vs `run_benchmark`'s built-in baselines (Gaussian / GCN naive propagation, TrivialSeed, the `null` no-change floor). The three expression spaces (observed raw counts / CONCERT raw / SpatialProp normalize_total) are unified by **per-cell `row_normalize`** so both metrics are scale-clean and the **one frozen cell annotator** (below) applies identically to every method. Run on the lab server, spatialprop env.

In [ ]:
%matplotlib inline
import sys, os, numpy as np, h5py
sys.path.insert(0, os.path.expanduser('~/spatial-pert/repo'))
import matplotlib.pyplot as plt
from spbench.adapters import SaundersAdapter
from spbench.config import run_benchmark
from spbench.external import row_normalize, score_external_seed
from spbench.models.spatialprop_model import SpatialPropModel
from spbench.models.concert_model import ConcertModel
from spbench.annotators import get_annotator, apply_annotation, annotation_fidelity
from spbench.graph import build_knn_graph
from spbench.composition import niche_effect_board

GUIDES = ['Hnf4a','Ldlr','Insr','Srebf1','B2m','Dnajb9','Dut','Hspe1','Hyou1','Tap1','Xbp1']
SP  = os.path.expanduser('~/spatial-pert/outputs/spatialprop/dumps_Saunders_b10')
CON = os.path.expanduser('~/spatial-pert/outputs/concert/out_b10')
spaths = {P: f'{SP}/{P}.h5ad' for P in GUIDES}
cpaths = {P: f'{CON}/saunders_b10_map_keep_{P}_perturbed_counts.h5ad' for P in GUIDES}

In [ ]:
def read_layer(p, L):
    with h5py.File(p, 'r') as f: return np.asarray(f['layers'][L], float)
def read_X(p):
    with h5py.File(p, 'r') as f:
        x = f['X']
        if isinstance(x, h5py.Dataset): return np.asarray(x, float)
        from scipy.sparse import csr_matrix
        return csr_matrix((np.asarray(x['data']), np.asarray(x['indices']), np.asarray(x['indptr'])),
                          shape=tuple(int(s) for s in x.attrs['shape'])).toarray().astype(float)

class NormNiche:  # per-cell normalize external niche to data.X space (PCC scale-invariant)
    def __init__(self, base): self.base = base
    def fit(self, *a, **k): return self
    def predict_niche(self, data, p, edges):
        nb = self.base.predict_niche(data, p, edges); return row_normalize(nb) if len(nb) else nb
    def predict(self, *a, **k): return self.base.predict(*a, **k)

## Load Saunders b10
`data.X` is set to `row_normalize(raw counts)`; SpatialProp/CONCERT niche predictions are wrapped with `NormNiche` so all three live in one normalize_total space.

In [ ]:
data = SaundersAdapter(os.path.expanduser('~/spatial-pert/inputs/saunders_b10_slice'),
                       max_files=1, counts_layer='X').load()
data.X = row_normalize(data.meta['counts'])
print('cells', data.n_cells, '| genes', data.n_genes, '| cell types', len(set(data.cell_type)))

## Cell annotation — the unified instrument

The composition Overlap needs **cell-type labels** for the *observed* niche AND each model's *predicted* niche. The rule that keeps the comparison fair: **label both sides with ONE frozen annotator**, so the annotator's own systematic error cancels and Overlap reflects *model* error, not labeling error. (A model is never scored against the dataset's expert labels directly — that would conflate the annotator's error with the model's.)

We use a **marker-score annotator** (reference-free, deterministic, pure-numpy):
1. **Fit** on the observed cells — from the dataset's own cell-type labels, derive the top marker genes per type (a `rank_genes_groups`-style standardized mean difference) and freeze the per-gene mean/σ. No model training, no train/test leakage.
2. **Predict** — z-score a cell with the frozen stats, score each cell-type's marker set, assign the arg-max type. The **same frozen instrument** is applied to observed and predicted expression.

Why marker here (not CellTypist/scANVI): identical across datasets, deterministic, and robust on model-*predicted* (off-manifold) expression, where a learned classifier is unreliable. The expert labels are used to *build* the instrument, not to score models.

**Fidelity** = agreement between `marker(observed expression)` and the dataset's expert labels — a one-off QC of the instrument (reported once; never on the model leaderboard). We then relabel `data.cell_type` with this frozen annotator so the descriptive board and the predictive Overlap share one labeling.

In [ ]:
ann = get_annotator('marker', n_markers=20).fit(data.X, data.cell_type, gene_names=data.gene_names)
fid = annotation_fidelity(ann, data.X, data.cell_type, gene_names=data.gene_names)
print('cell types :', list(ann.cats_))
print(f'fidelity   : {fid:.3f}  (marker(observed) vs dataset expert labels; 1.0 = exact)')
data, ann = apply_annotation(data, ann)   # relabel data.cell_type with the SAME frozen instrument
print('annotator  :', data.meta['annotator'])

## Run the benchmark
`annotator=ann` is threaded through, so every method's predicted niche is labelled by the one frozen instrument and a composition **Overlap** is returned next to **PCC-delta** in `res['compare']`.

In [ ]:
res = run_benchmark(data, GUIDES, k=15, gcn_kwargs={'hidden':64,'epochs':30}, progress=False,
                    annotator=ann,
                    external_models={'SpatialProp': NormNiche(SpatialPropModel(spaths)),
                                     'CONCERT': NormNiche(ConcertModel(cpaths))})

## niche PCC-delta (bystander neighbours; higher = better)

In [ ]:
print(f"{'guide':8} {'n':>4} | {'Gauss':>7} {'GCN':>7} | {'SpatProp':>8} {'CONCERT':>8}")
nA = {'g':[], 'gc':[], 'sp':[], 'con':[]}
for P in GUIDES:
    pcc = res['compare'][P]['pcc']; n = int((data.perturbation==P).sum())
    g,gc,sp,con = (pcc.get(k, np.nan) for k in ('model+base','model+learned','SpatialProp','CONCERT'))
    for k,v in zip(nA, (g,gc,sp,con)): nA[k].append(v)
    print(f'{P:8} {n:4d} | {g:>7.3f} {gc:>7.3f} | {sp:>8.3f} {con:>8.3f}')
print(f"{'MEAN':8} {'':>4} | {np.nanmean(nA['g']):>7.3f} {np.nanmean(nA['gc']):>7.3f} | "
      f"{np.nanmean(nA['sp']):>8.3f} {np.nanmean(nA['con']):>8.3f}")

## niche composition Overlap (cell-type makeup; higher = better)

Same predicted niches as the PCC table, different question: instead of *expression direction*, **does the predicted niche have the right cell-type composition?** Each method's predicted bystander niche is labelled by the frozen marker instrument and its composition compared to the observed niche's via `Overlap = 1 - TV`. `null` is the predict-no-change baseline (the control niche) — **a method is only useful if it beats `null`**.

In [ ]:
print(f"{'guide':8} {'n':>4} | {'Gauss':>7} {'GCN':>7} | {'SpatProp':>8} {'CONCERT':>8} | {'null':>6}")
oA = {'g':[], 'gc':[], 'sp':[], 'con':[], 'null':[]}
for P in GUIDES:
    ov = res['compare'][P]['overlap']; n = int((data.perturbation==P).sum())
    g,gc,sp,con,nu = (ov.get(k, np.nan) for k in ('model+base','model+learned','SpatialProp','CONCERT','null'))
    for k,v in zip(oA, (g,gc,sp,con,nu)): oA[k].append(v)
    print(f'{P:8} {n:4d} | {g:>7.3f} {gc:>7.3f} | {sp:>8.3f} {con:>8.3f} | {nu:>6.3f}')
print(f"{'MEAN':8} {'':>4} | {np.nanmean(oA['g']):>7.3f} {np.nanmean(oA['gc']):>7.3f} | "
      f"{np.nanmean(oA['sp']):>8.3f} {np.nanmean(oA['con']):>8.3f} | {np.nanmean(oA['null']):>6.3f}")

## seed PCC-delta (perturbed centre cell; higher = better)

In [ ]:
print(f"{'guide':8} {'n':>4} | {'TrivSeed':>8} | {'SpatProp':>8} {'CONCERT':>8}")
sA = {'ts':[], 'sp':[], 'con':[]}
for P in GUIDES:
    ts = res['seed'][P]['pcc_delta']
    sp = score_external_seed(data, P, row_normalize(read_layer(spaths[P],'predicted_tempered')))['pcc_delta']
    con = score_external_seed(data, P, row_normalize(read_X(cpaths[P])))['pcc_delta']
    n = int((data.perturbation==P).sum())
    for k,v in zip(sA, (ts,sp,con)): sA[k].append(v)
    print(f'{P:8} {n:4d} | {ts:>8.3f} | {sp:>8.3f} {con:>8.3f}')
print(f"{'MEAN':8} {'':>4} | {np.nanmean(sA['ts']):>8.3f} | "
      f"{np.nanmean(sA['sp']):>8.3f} {np.nanmean(sA['con']):>8.3f}")

## Descriptive: which guides actually reshape the niche?

Before asking models to *predict* the effect, quantify the **observed** effect per guide — the ground-truth / TARDIS-style readout, computed with our own Overlap on the unified labels. Two baselines:
- **ovNTC** — KO-cell niche vs the control niche (raw effect vs unperturbed);
- **ovPool** — KO-cell niche vs the *average perturbed* niche → the **guide-specific** shift, de-confounding any shared 'where do guide-bearing cells sit' axis.

`p` = label-permutation test (shuffle KO vs control, keep n_ko; N=1000). Lower Overlap = bigger niche reshaping; rows sorted by the guide-specific shift, with the cell types that move most.

In [ ]:
edges = build_knn_graph(data, k=15)
board = niche_effect_board(data, edges, min_ko=1, n_perm=1000, seed=0, perturbations=GUIDES)
cats = board['cats']
print(f"{'guide':8}{'nKO':>5}{'ovNTC':>7}{'ovPool':>8}{'p':>7}   top guide-specific shifts (pooled delta %)")
for r in board['rows']:
    d = np.asarray(r['delta_pooled']); idx = np.argsort(-np.abs(d))[:3]
    sh = '  '.join(f'{cats[i]}({d[i]*100:+.1f})' for i in idx)
    print(f"{r['guide']:8}{r['n_ko']:5d}{r['overlap_ntc']:7.3f}{r['overlap_pooled']:8.3f}"
          f"{r.get('p_value', float('nan')):7.3f}   {sh}")

## PCC-delta heatmap (niche + seed)

In [ ]:
def heat(ax, d, title):
    rows = list(d); cols = GUIDES + ['MEAN']
    M = np.array([d[r] + [float(np.nanmean(d[r]))] for r in rows])
    im = ax.imshow(M, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=40, ha='right', fontsize=9)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=10)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f'{M[i,j]:.2f}', ha='center', va='center', fontsize=7.5,
                    color='white' if abs(M[i,j])>0.6 else 'black', fontweight='bold' if j==len(GUIDES) else 'normal')
    ax.axvline(len(GUIDES)-0.5, color='k', lw=1.5); ax.set_title(title, fontsize=12, loc='left', fontweight='bold')
    return im
niche_d = {'Gaussian':nA['g'],'GCN':nA['gc'],'SpatialProp':nA['sp'],'CONCERT':nA['con']}
seed_d  = {'TrivialSeed':sA['ts'],'SpatialProp':sA['sp'],'CONCERT':sA['con']}
fig, axes = plt.subplots(2, 1, figsize=(15, 7.5), gridspec_kw={'height_ratios':[4,3]})
heat(axes[0], niche_d, 'NICHE  PCC-delta  (bystander neighbours)')
im = heat(axes[1], seed_d, 'SEED  PCC-delta  (perturbed centre cell)')
fig.suptitle('Saunders b10 · PCC-delta benchmark (11 in-panel guides) — higher = better, 0 = no skill',
             fontsize=13, fontweight='bold')
fig.colorbar(im, ax=axes, shrink=0.55, label='PCC-delta', pad=0.015)
out = os.path.expanduser('~/spatial-pert/outputs/figures/pcc_compare_b10.png')
os.makedirs(os.path.dirname(out), exist_ok=True); fig.savefig(out, dpi=140, bbox_inches='tight')
plt.show(); print('saved', out)